code Python  porte sur le nettoyage et le prétraitement du jeu de données  avant l’entraînement d’un modèle de machine learning.

1. Importer les bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")


2. Charger le jeu de données

In [ ]:
df = pd.read_csv("Life Expectancy Data.csv")#
df = pd.read_csv("data/Life Expectancy Data.csv")#Si le fichier n’est pas dans le même dossier que le programme, indiquez son chemin
df= #telecharger directement a partir d un site web(kaggle,...)

In [ ]:
print(df.head())
print(df.tail())

3. Vérifier les données

In [ ]:
# Dimensions du jeu de données
print(df.shape)

# Informations générales
df.info()

# Valeurs manquantes
print(df.isnull().sum())

# Pourcentage de valeurs manquantes
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage)

# Doublons
print("Nombre de doublons :", df.duplicated().sum())


vérifie également les valeurs des colonnes catégorielles :

In [ ]:
for column in df.select_dtypes(include="object").columns:
    print(f"\nColonne : {column}")
    print(df[column].value_counts())
    print("*" * 30)


4. Analyse exploratoire des données

Statistiques descriptives

In [ ]:
# Colonnes numériques
print(df.describe().T)

# Colonnes catégorielles
print(df.describe(include="object"))


Histogrammes

In [ ]:
for column in df.select_dtypes(include="number").columns:
    sns.histplot(data=df, x=column, kde=True)
    plt.title(f"Distribution de {column}")
    plt.show()


Boxplots pour repérer les valeurs aberrantes

In [ ]:
for column in df.select_dtypes(include="number").columns:
    sns.boxplot(data=df, x=column)
    plt.title(f"Valeurs aberrantes : {column}")
    plt.show()


Nuages de points avec l’espérance de vie

In [ ]:
columns_to_plot = [
    "Year",
    "Adult Mortality",
    "Infant deaths",
    "Alcohol",
    "Hepatitis B",
    "BMI",
    "GDP",
    "Population"
]

for column in columns_to_plot:
    sns.scatterplot(
        data=df,
        x=column,
        y="Life expectancy"
    )
    plt.title(f"{column} et espérance de vie")
    plt.show()


Matrice de corrélation

In [ ]:
correlation = df.select_dtypes(include="number").corr()

plt.figure(figsize=(15, 15))
sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Matrice de corrélation")
plt.show()


5. Traiter les valeurs manquantes

certaines colonnes sont complétées avec leur médiane :

In [ ]:
columns_to_impute = [
    "BMI",
    "Polio",
    "Income composition of resources"
]

for column in columns_to_impute:
    df[column] = df[column].fillna(df[column].median())


utilise également KNNImputer pour les colonnes numériques :

In [ ]:
from sklearn.impute import KNNImputer

numeric_columns = df.select_dtypes(include="number").columns

imputer = KNNImputer()

df[numeric_columns] = imputer.fit_transform(
    df[numeric_columns]
)


Vérification finale :

Remarque : appliquer KNNImputer à toutes les colonnes numériques est plus cohérent que de l’appliquer séparément à chaque colonne. Dans cette dernière situation, il revient pratiquement à remplacer les valeurs manquantes par une valeur centrale.

In [ ]:
print(df.isnull().sum())


6. Traiter les valeurs aberrantes
on définit une fonction basée sur l’intervalle interquartile, ou IQR

In [ ]:
def whisker_limits(column):
    q1 = np.percentile(column, 25)
    q3 = np.percentile(column, 75)

    iqr = q3 - q1

    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr

    return lower_limit, upper_limit


Les valeurs extrêmes sont ensuite écrêtées

In [ ]:
outlier_columns = [
    "GDP",
    "Total expenditure",
    "Thinness 1-19 years",
    "Thinness 5-9 years"
]

for column in outlier_columns:
    lower_limit, upper_limit = whisker_limits(df[column])

    df[column] = np.where(
        df[column] < lower_limit,
        lower_limit,
        df[column]
    )

    df[column] = np.where(
        df[column] > upper_limit,
        upper_limit,
        df[column]
    )


Vérification avec des boxplots :

In [ ]:
for column in outlier_columns:
    sns.boxplot(data=df, x=column)
    plt.title(f"Après traitement : {column}")
    plt.show()


7. Supprimer les doublons

In [ ]:
df.drop_duplicates(inplace=True)

print("Nombre de doublons après suppression :")
print(df.duplicated().sum())


8. Encoder les variables catégorielles
Les colonnes Country et Status sont transformées en variables binaires :

In [ ]:
df_encoded = pd.get_dummies(
    data=df,
    columns=["Country", "Status"],
    drop_first=True
)


Vérifier le résultat :

In [ ]:
print(df_encoded.head())
print(df_encoded.shape)


Après cette étape, df_encoded contient des données numériques nettoyées et encodées, prêtes à être séparées en variables explicatives X et variable cible y pour entraîner un modèle.

**scinder un jeu de données en ensembles d'entraînement et de test pour y appliquer un algorithme de régression linéaire ?**


Pour appliquer une régression linéaire, vous devez séparer :
X : les variables explicatives ;
y : la variable cible, ici Life expectancy.
Puis diviser les données en deux ensembles :
entraînement : utilisé pour apprendre le modèle ;
test : utilisé pour mesurer ses performances sur des données jamais vues.

1. Séparation simple avec df_encoded
Si votre jeu de données final s’appelle df_encoded :

In [ ]:
from sklearn.model_selection import train_test_split

# Variable cible
y = df_encoded["Life expectancy"]

# Variables explicatives
X = df_encoded.drop(columns=["Life expectancy"])

# Séparation : 80 % entraînement, 20 % test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)#random_state=42 permet d’obtenir la même séparation à chaque exécution

print("Taille de X_train :", X_train.shape)
print("Taille de X_test  :", X_test.shape)
print("Taille de y_train :", y_train.shape)
print("Taille de y_test  :", y_test.shape)


2. Entraîner une régression linéaire

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

# Apprentissage uniquement sur l'ensemble d'entraînement
model.fit(X_train, y_train)


3. Faire des prédictions\n
y_pred contient les espérances de vie prédites par le modèle pour l’ensemble de test.

In [ ]:
y_pred = model.predict(X_test)

print(y_pred[:10])


4. Évaluer le modèle
Utilisez plusieurs métriques :

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE  :", mae)
print("MSE  :", mse)
print("RMSE :", rmse)
print("R²   :", r2)


Interprétation
MAE : erreur absolue moyenne, exprimée dans l’unité de la cible, ici des années ;
RMSE : pénalise davantage les grandes erreurs ;
R² : proportion de la variation expliquée par le modèle.
Par exemple, un MAE de 1.8 signifie que les prédictions diffèrent en moyenne de 1,8 année de la valeur réelle.

5. Visualiser les prédictions

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_pred, alpha=0.6)

# Droite idéale : prédiction = valeur réelle
minimum = min(y_test.min(), y_pred.min())
maximum = max(y_test.max(), y_pred.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    color="red",
    linestyle="--"
)

plt.xlabel("Espérance de vie réelle")
plt.ylabel("Espérance de vie prédite")
plt.title("Valeurs réelles et valeurs prédites")
plt.show()
#Plus les points sont proches de la droite rouge, meilleures sont les prédictions.

6. Examiner les coefficients

Les coefficients indiquent l’influence de chaque variable :

In [ ]:
coefficients = pd.DataFrame({
    "variable": X.columns,
    "coefficient": model.coef_
})

coefficients["valeur_absolue"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    by="valeur_absolue",
    ascending=False
)

print(coefficients)


L’ordonnée à l’origine est disponible avec :

Attention : les coefficients ne sont directement comparables que si les variables sont sur des échelles similaires.

In [ ]:
print("Intercept :", model.intercept_)


7. Version recommandée avec une validation correcte
Dans votre code précédent, l’imputation et le traitement des valeurs aberrantes ont été effectués avant la séparation des données. Cela peut provoquer une fuite d’information (data leakage) : certaines informations provenant de l’ensemble de test ont indirectement servi à préparer les données d’entraînement.
Une méthode plus rigoureuse consiste à séparer d’abord les données, puis à ajuster les traitements uniquement sur X_train.

Exemple avec une pipeline

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


Supposons que df soit le jeu de données original, avant l’encodage :

In [ ]:
# Nettoyage des noms de colonnes
df.columns = df.columns.str.strip()

# Suppression des doublons
df = df.drop_duplicates()

# Séparation de la cible et des variables
X = df.drop(columns=["Life expectancy"])
y = df["Life expectancy"]

# Identifier les colonnes numériques et catégorielles
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

# Prétraitement des colonnes numériques
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Prétraitement des colonnes catégorielles
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        drop="first"
    ))
])

# Préprocesseur global
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Pipeline complète
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("regression", LinearRegression())
])


Puis on entraîne la pipeline :

In [ ]:
pipeline.fit(X_train, y_train)


Et on évalue le modèle :

In [ ]:
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.3f}")


Cette approche est préférable, car :
la médiane est calculée uniquement à partir de X_train ;
l’encodage est appris uniquement sur X_train ;
les nouvelles catégories présentes dans X_test sont gérées avec handle_unknown="ignore" ;
le même prétraitement est appliqué automatiquement à l’entraînement et au test.

Exemple complet minimal
Si vous utilisez déjà df_encoded, le code essentiel est :

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

X = df_encoded.drop(columns=["Life expectancy"])
y = df_encoded["Life expectancy"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R² :", r2_score(y_test, y_pred))


Pour une première expérimentation, cette version suffit. Pour une analyse plus fiable, utilisez la pipeline, qui évite les fuites d’information pendant le prétraitement